# Case B / 案例 B：Retail sales regression / 零售销量回归
This case uses two fixed CSV files with the same schema and business meaning: `case-b-retail-teaching.csv` contains D1–D8 for hand calculation, while `case-b-retail.csv` contains a 4,800-row product–store–day panel for model comparison. The generator verifies both files byte for byte. / 本案例使用两份字段语义一致的固定 CSV：`case-b-retail-teaching.csv` 包含用于手算的 D1—D8，`case-b-retail.csv` 包含 4,800 条用于模型对照的商品—门店—日记录。Notebook 会逐字节核对两份文件。

In [ ]:
import numpy as np
import pandas as pd
import sklearn, xgboost, lightgbm, catboost
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from pathlib import Path
import sys
bundle = Path('content/posts/2026-10-17-from-a-single-tree-to-a-forest-mathematical-principles-algorithmic-evolution-and-engineering-practices-of-tree-models')
if not bundle.is_dir():
    bundle = Path.cwd()
sys.path.insert(0, str(bundle.resolve()))
from generate_tree_cases import generate_retail, generate_retail_teaching
teaching_path = bundle / 'case-b-retail-teaching.csv'
csv_path = bundle / 'case-b-retail.csv'
assert teaching_path.read_text(encoding='utf-8') == generate_retail_teaching().to_csv(index=False, float_format='%.12g')
assert csv_path.read_text(encoding='utf-8') == generate_retail().to_csv(index=False, float_format='%.12g')
toy = pd.read_csv(teaching_path).rename(columns={'record_id':'id','lag7':'past_sales'}).sort_values('id').reset_index(drop=True)
panel = pd.read_csv(csv_path)
assert len(toy) == 8 and len(panel) == 4800 and panel.record_id.is_unique
print('Versions / 版本:', sklearn.__version__, xgboost.__version__, lightgbm.__version__, catboost.__version__)


## 1. Hand calculations on D1–D8 / D1—D8 手算
For the squared loss `0.5*(y-z)^2`, the initial prediction is explicitly set to 50; it is not claimed to be a library default. / 采用平方损失并显式设置初始预测 50 件，不将其称作软件默认值。

In [ ]:
base, lam, gamma, eta = 50.0, 1.0, 100.0, 0.3
toy['g1'] = base - toy.y; toy['h1'] = 1.0
def stats(frame):
    return float(frame.g1.sum()), float(frame.h1.sum())
def gain(parent, left, right):
    g,h=stats(parent); gl,hl=stats(left); gr,hr=stats(right)
    return 0.5*(gl**2/(hl+lam)+gr**2/(hr+lam)-g**2/(h+lam))-gamma
left=toy[toy.discount<=25]; right=toy[toy.discount>25]
ll=left[left.discount<=15]; lr=left[left.discount>15]
rl=right[right.discount<=35]; rr=right[right.discount>35]
root_gain=gain(toy,left,right); left_gain=gain(left,ll,lr); right_gain=gain(right,rl,rr)
assert len(toy)==8 and toy.y.sum()==400 and stats(toy)==(0.0,8.0)
assert np.isclose(root_gain,4100.833333333333)
assert np.isclose(left_gain,-12.933333333333)
assert np.isclose(right_gain,308.5)
weights={'L':-stats(left)[0]/(stats(left)[1]+lam),'RL':-stats(rl)[0]/(stats(rl)[1]+lam),'RR':-stats(rr)[0]/(stats(rr)[1]+lam)}
assert np.allclose(list(weights.values()),[-142/6,5,44])
toy['leaf']=np.select([toy.id.isin(left.id),toy.id.isin(rl.id)],['L','RL'],default='RR')
toy['yhat1']=base+eta*toy.leaf.map(weights)
toy['g2']=toy.yhat1-toy.y; toy['h2']=1.0
root_sse=((toy.y-50)**2).sum()
cart_sse=((left.y-left.y.mean())**2).sum()+((right.y-right.y.mean())**2).sum()
gbdt_pred=np.where(toy.discount<=25,50+eta*(left.y.mean()-50),50+eta*(right.y.mean()-50))
xgb_root_pred=np.where(toy.discount<=25,50+eta*(-142/6),50+eta*(142/4))
assert np.isclose(root_sse,14560) and np.isclose(cart_sse,3805.866666666667)
assert np.isclose(((toy.y-gbdt_pred)**2).sum(),9075.392)
assert np.isclose(((toy.y-xgb_root_pred)**2).sum(),10111.3175)
print('CART/GBDT/XGBoost root SSE / 根节点计算:',round(cart_sse,2),round(((toy.y-gbdt_pred)**2).sum(),2),round(((toy.y-xgb_root_pred)**2).sum(),2))
print('Net gains / 净增益:',round(root_gain,2),round(left_gain,2),round(right_gain,2))
print('Leaf weights / 叶权重:',weights)
print(toy[['id','discount','past_sales','promo_type','y','g1','h1','leaf','yhat1','g2']].round(2).to_string(index=False))


In [ ]:
def all_candidates(frame):
    found=[]
    for col in ['discount','past_sales']:
        values=sorted(frame[col].unique())
        for a,b in zip(values[:-1],values[1:]):
            cut=(a+b)/2; lo=frame[frame[col]<=cut]; hi=frame[frame[col]>cut]
            found.append((gain(frame,lo,hi),col,cut))
    return sorted(found,reverse=True)
assert np.isclose(all_candidates(left)[0][0],left_gain)
assert np.isclose(all_candidates(right)[0][0],right_gain)
assert np.isclose(all_candidates(toy)[0][0],root_gain)
discount_gains=[gain(toy,toy[toy.discount<=cut],toy[toy.discount>cut]) for cut in [5,15,25,35,45]]
assert np.allclose(discount_gains,[1275.2380952380952,3384.8,4100.833333333333,4048.571428571428,2431.25])
print('Discount net gains / 折扣候选净增益:',np.round(discount_gains,2))
print('Best left and right candidates / 左右节点最大候选增益:',all_candidates(left)[0],all_candidates(right)[0])


## 2. Histogram and leaf choice / 直方图与叶节点选择
The three bins below are fixed for teaching; they are not the exact bin boundaries a fitted library must choose. / 下列三桶为教学指定，不代表软件必然选择的桶边界。XGBoost can also use histograms. / XGBoost 同样可以使用直方图。

In [ ]:
toy['demo_bin']=pd.cut(toy.discount,[-1,20,35,100],labels=['0–20','30','40–50'])
bins=toy.groupby('demo_bin',observed=True).agg(G=('g1','sum'),H=('h1','sum'))
assert np.allclose(bins.G,[142,-10,-132]) and np.allclose(bins.H,[5,1,2])
print(bins.to_string())
print('After the root / 根节点之后: left net gain',round(left_gain,2),'right net gain',round(right_gain,2))
print('Only the right child has positive net gain under these assumptions. / 此设定下仅右子节点值得继续切分。')


## 3. Ordered-statistic illustration / 有序统计示意
The fixed-order smoothed mean excludes each row's own target. It is **not** the full CatBoost CTR or Ordered Boosting algorithm. / 固定排列的平滑均值排除样本自身标签，但不是 CatBoost 完整的 CTR 或 Ordered Boosting。

In [ ]:
prior,strength=50.0,1.0; seen={}; ordered=[]
for row in toy.itertuples():
    total,count=seen.get(row.promo_type,(0.0,0))
    ordered.append((total+strength*prior)/(count+strength))
    seen[row.promo_type]=(total+row.y,count+1)
toy['ordered_demo']=ordered
toy['full_mean_leaky']=toy.groupby('promo_type').y.transform('mean')
assert np.isclose(toy.loc[4,'ordered_demo'],50.0) and np.isclose(toy.loc[5,'ordered_demo'],45.0)
print(toy[['id','promo_type','y','ordered_demo','full_mean_leaky']].round(2).to_string(index=False))


## 4. Chronological partitions of the fixed CSV / 固定 CSV 的时间划分
Planned promotion and discount are known before the forecast. `lag7` uses only earlier sales. / 促销及折扣为事先计划，`lag7` 只用预测日前销量。Real promotional peaks are retained; only selected training labels receive artificial recording errors. / 真实促销峰值保留，仅污染选定训练标签。

In [ ]:
train=panel.loc[panel.split=='train'].copy()
valid=panel.loc[panel.split=='valid'].copy()
test=panel.loc[panel.split=='test'].copy()
assert set(toy.id).isdisjoint(set(panel.record_id))
assert 'teaching_row' not in panel.columns
assert len(train)==2920 and len(valid)==len(test)==800
assert train.day.max()<valid.day.min()<test.day.min()
first=panel[(panel.sku_id=='SKU00')&(panel.store_id=='ST0')&(panel.day==7)].iloc[0]
history=panel[(panel.sku_id=='SKU00')&(panel.store_id=='ST0')&(panel.day<7)]
assert first.day==7 and np.isfinite(first.lag7)
error_rng=np.random.default_rng(142)
dirty=train.y.astype(float).copy()
bad_index=error_rng.choice(train.index,size=round(.03*len(train)),replace=False)
dirty.loc[bad_index]+=error_rng.choice([-1,1],len(bad_index))*5*train.y.std()
assert len(bad_index)==88 and (dirty.loc[train.index.difference(bad_index)]==train.y.loc[train.index.difference(bad_index)]).all()
print('Rows / 样本量:',len(train),len(valid),len(test),'corrupted training labels / 污染训练标签:',len(bad_index))


In [ ]:
raw_panel=panel
expected=raw_panel[(raw_panel.sku_id=='SKU00')&(raw_panel.store_id=='ST0')&(raw_panel.day<7)].y.mean()
assert np.isclose(first.lag7,expected)  # day 7 excludes its own target / 第 7 天不使用自身销量


## 5. Shared holdout comparison / 同一测试集对照
All three boosting libraries receive the same semantic fields through their native category APIs. The two scikit-learn baselines use one-hot columns fitted on training data only. / 三种提升树共享字段语义；scikit-learn 基线仅以训练集拟合 One-hot 编码。The test labels are never corrupted. / 测试标签从不污染。

In [ ]:
features=['discount','weekday','lag7','sku_id','store_id','promo_type']
cat_cols=['sku_id','store_id','promo_type']
cat_levels={c:sorted(train[c].unique()) for c in cat_cols}
def frame(part,family):
    x=part[features].copy()
    for c,levels in cat_levels.items():
        x[c]=x[c].astype(pd.CategoricalDtype(levels)) if family!='CatBoost' else x[c].astype(str)
    return x
encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
encoder.fit(train[cat_cols])
def baseline_frame(part):
    return np.column_stack([part[['discount','weekday','lag7']].to_numpy(),encoder.transform(part[cat_cols])])
def metrics(y,pred):
    err=pred-y.to_numpy(); den=np.abs(y).sum()
    return [mean_absolute_error(y,pred),mean_squared_error(y,pred)**.5,np.abs(err).sum()/den,err.sum()/y.sum()]
rows=[]
clean_xgb_pred = None
for setting,target in [('clean',train.y),('dirty',dirty)]:
    models={
        'Tree':DecisionTreeRegressor(max_depth=6,random_state=42),
        'Random Forest':RandomForestRegressor(n_estimators=100,max_depth=10,max_features=.7,n_jobs=1,random_state=42),
        'XGBoost':XGBRegressor(n_estimators=100,max_depth=4,learning_rate=.06,tree_method='hist',enable_categorical=True,n_jobs=1,random_state=42),
        'LightGBM':LGBMRegressor(n_estimators=100,num_leaves=15,min_child_samples=20,learning_rate=.06,verbosity=-1,n_jobs=1,random_state=42),
        'CatBoost':CatBoostRegressor(iterations=100,depth=4,learning_rate=.06,boosting_type='Ordered',loss_function='RMSE',cat_features=cat_cols,thread_count=1,random_seed=42,verbose=False,allow_writing_files=False),
    }
    for name,model in models.items():
        if name in ['Tree','Random Forest']:
            model.fit(baseline_frame(train),target); pred=model.predict(baseline_frame(test))
        else:
            model.fit(frame(train,name),target); pred=model.predict(frame(test,name))
        if name=='XGBoost' and setting=='clean': clean_xgb_pred=pred.copy()
        rows.append([name,setting,*metrics(test.y,pred)])
results_b=pd.DataFrame(rows,columns=['model','setting','MAE','RMSE','WAPE','ForecastBias'])
print(results_b.round(3).to_string(index=False))
err = clean_xgb_pred-test.y.to_numpy()
print('Clean XGBoost test aggregates / 干净训练的测试集汇总:',len(test),round(float(test.y.sum()),2),round(float(np.abs(err).sum()),2),round(float(np.square(err).sum()),2),round(float(err.sum()),2))
print(test[['record_id','business_date','y']].head(3).assign(prediction=np.round(clean_xgb_pred[:3],2)).to_string(index=False))
